# Proyek Pengembangan Machine Learning Pipeline: Heart Disease Prediction
- **Nama:** M. Rizal Basri
- **Username Dicoding:** rizalbasri
- **Dataset:** Heart Disease Dataset (UCI Machine Learning Repository / Cleveland)
- **Pipeline Name:** rizalbasri-pipeline
- **Framework:** TensorFlow Extended (TFX) dengan InteractiveContext

---
## Deskripsi Proyek
Proyek ini mengimplementasikan Machine Learning Pipeline end-to-end berstandar industri menggunakan **TensorFlow Extended (TFX)**. Seluruh komponen pipeline dijalankan secara interaktif menggunakan `InteractiveContext` untuk menyelesaikan permasalahan klasifikasi risiko penyakit jantung (*Heart Disease Prediction*).

### Komponen TFX yang Diimplementasikan (Lengkap dengan Saran Bintang 5):
1. **CsvExampleGen**: Ingest data dari berkas CSV dan membaginya menjadi split training & evaluation berformat TFRecord.
2. **StatisticsGen**: Menghitung ringkasan statistik deskriptif dari dataset menggunakan TensorFlow Data Validation (TFDV).
3. **SchemaGen**: Menginferensikan skema data (tipe data, domain nilai, fitur wajib) dari statistik data.
4. **ExampleValidator**: Memvalidasi data terhadap skema dan mendeteksi anomali atau data drift.
5. **Transform**: Melakukan rekayasa fitur (feature engineering & scaling) menggunakan TensorFlow Transform (TFT) agar konsisten saat training dan serving.
6. **Tuner (Saran 1 - Bintang 5)**: Menjalankan hyperparameter tuning otomatis dengan KerasTuner (RandomSearch) untuk memilih konfigurasi arsitektur dan learning rate terbaik.
7. **Trainer**: Melatih model Deep Neural Network (DNN) menggunakan Keras dengan `best_hyperparameters` hasil Tuner dan mengekspor model bertanda tangan serving (`serving_default`).
8. **Resolver (LatestBlessedModelResolver)**: Menentukan model terbaik sebelumnya sebagai acuan baseline komparasi.
9. **Evaluator**: Mengevaluasi performa model menggunakan TensorFlow Model Analysis (TFMA) dengan slicing spesifik (fitur jenis kelamin `sex`) dan memvalidasi threshold blessing.
10. **Pusher**: Memverifikasi model yang telah di-bless dan mendistribusikannya ke direktori serving siap produksi.


## 1. Import Library dan Menyiapkan Direktori
Langkah pertama adalah mengimpor pustaka utama yang dibutuhkan seperti TensorFlow, TFX, TensorFlow Data Validation, TensorFlow Transform, Keras Tuner, dan TensorFlow Model Analysis, serta menentukan direktori pipeline dan serving model.


In [1]:
import os
import shutil
import pandas as pd
import tensorflow as tf
from tfx.orchestration.experimental.interactive.interactive_context import InteractiveContext

print(f"TensorFlow Version: {tf.__version__}")
import tfx
print(f"TFX Version: {tfx.__version__}")

PIPELINE_NAME = "rizalbasri-pipeline"
PIPELINE_ROOT = os.path.join("pipeline_root")
SERVING_MODEL_DIR = os.path.join("serving_model_dir")
DATA_ROOT = os.path.join("data")

context = InteractiveContext(pipeline_name=PIPELINE_NAME, pipeline_root=PIPELINE_ROOT)


D:\Coding\dicoding\M_Rizal_Basri-pipeline\.venv\lib\site-packages\google\api_core\_python_version_support.py:275: FutureWarning: You are using a Python version (3.10.21) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)


D:\Coding\dicoding\M_Rizal_Basri-pipeline\.venv\lib\site-packages\google\auth\transport\grpc.py:44: FutureWarning: grpcio < 1.83.0 does not support Post-Quantum Cryptography (PQC). Support for non-PQC environments is deprecated. In October 2026, google-auth will raise its minimum requirements to enforce grpcio >= 1.83.0. For more details on Google Cloud's post-quantum security migration, visit: https://cloud.google.com/security/resources/post-quantum-cryptography
  warnings.warn(
D:\Coding\dicoding\M_Rizal_Basri-pipeline\.venv\lib\site-packages\google\api_core\_python_version_support.py:275: FutureWarning: You are using a Python version (3.10.21) which Google will stop supporting in new releases of google.cloud.bigquery_storage_v1 once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.cloud.bigquery_storage_v1 past that date.
  warnings.warn(message, FutureWarning)


D:\Coding\dicoding\M_Rizal_Basri-pipeline\.venv\lib\site-packages\google\api_core\_python_version_support.py:275: FutureWarning: You are using a Python version (3.10.21) which Google will stop supporting in new releases of google.pubsub_v1 once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.pubsub_v1 past that date.
  warnings.warn(message, FutureWarning)


TensorFlow Version: 2.13.1
TFX Version: 1.14.0


## 2. Eksplorasi Singkat Data Awal
Dataset yang digunakan adalah Heart Disease dataset yang memuat 13 fitur klinis (kombinasi data kontinu seperti umur, tekanan darah, kolesterol, detak jantung, serta data kategorikal seperti jenis kelamin, tipe nyeri dada, gula darah, dan hasil EKG) dan 1 label target klasifikasi biner.


In [2]:
df = pd.read_csv(os.path.join(DATA_ROOT, "heart.csv"))
print("Jumlah baris dan kolom:", df.shape)
df.head()


Jumlah baris dan kolom: (303, 14)


,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal,target
0,63,1,3,145,233,1,0,150,0,2.3,0,0,1,1
1,37,1,2,130,250,0,1,187,0,3.5,0,0,2,1
2,41,0,1,130,204,0,0,172,0,1.4,2,0,2,1
3,56,1,1,120,236,0,1,178,0,0.8,2,0,2,1
4,57,0,0,120,354,0,1,163,1,0.6,2,0,2,1


## 3. Komponen 1: ExampleGen (Data Ingestion)
`CsvExampleGen` bertugas membaca berkas CSV mentah dari direktori data, mengonversinya ke format biner yang dioptimalkan untuk TensorFlow (`tf.train.Example` dalam berkas TFRecord), dan secara otomatis membagi data menjadi split `train` dan `eval` (rasio default 2:1 atau 67:33).


In [3]:
from tfx.components import CsvExampleGen

example_gen = CsvExampleGen(input_base=DATA_ROOT)
context.run(example_gen)


ExecutionResult(
    component_id: CsvExampleGen
    execution_id: 11
    outputs:
        examples: OutputChannel(artifact_type=Examples, producer_component_id=CsvExampleGen, output_key=examples, additional_properties={}, additional_custom_properties={}, _input_trigger=None)

Menampilkan lokasi artefak TFRecord yang dihasilkan oleh `CsvExampleGen`:


In [4]:
train_uri = os.path.join(example_gen.outputs['examples'].get()[0].uri, 'Split-train')
eval_uri = os.path.join(example_gen.outputs['examples'].get()[0].uri, 'Split-eval')
print("Train TFRecord URI:", train_uri)
print("Eval TFRecord URI:", eval_uri)


Train TFRecord URI: pipeline_root\CsvExampleGen\examples\11\Split-train
Eval TFRecord URI: pipeline_root\CsvExampleGen\examples\11\Split-eval


## 4. Komponen 2: StatisticsGen (Data Statistics)
`StatisticsGen` menghitung ringkasan statistik deskriptif dari seluruh dataset (baik split `train` maupun `eval`) menggunakan TensorFlow Data Validation (TFDV). Hal ini berguna untuk melihat sebaran nilai, mendeteksi missing values, serta memeriksa kuantil data numerik dan kategorikal.


In [5]:
from tfx.components import StatisticsGen

statistics_gen = StatisticsGen(examples=example_gen.outputs['examples'])
context.run(statistics_gen)


ExecutionResult(
    component_id: StatisticsGen
    execution_id: 12
    outputs:
        statistics: OutputChannel(artifact_type=ExampleStatistics, producer_component_id=StatisticsGen, output_key=statistics, additional_properties={}, additional_custom_properties={}, _input_trigger=None)

Menampilkan visualisasi statistik data interaktif:


In [6]:
context.show(statistics_gen.outputs['statistics'])


## 5. Komponen 3: SchemaGen (Data Schema)
`SchemaGen` menginferensikan skema data secara otomatis berdasarkan statistik yang dihasilkan oleh `StatisticsGen`. Skema ini memuat tipe data yang diharapkan untuk setiap fitur (misal: INT, FLOAT), properti keberadaan (presence), serta batasan nilai (domain).


In [7]:
from tfx.components import SchemaGen

schema_gen = SchemaGen(
    statistics=statistics_gen.outputs['statistics'],
    infer_feature_shape=True
)
context.run(schema_gen)


ExecutionResult(
    component_id: SchemaGen
    execution_id: 13
    outputs:
        schema: OutputChannel(artifact_type=Schema, producer_component_id=SchemaGen, output_key=schema, additional_properties={}, additional_custom_properties={}, _input_trigger=None)

Menampilkan visualisasi skema data:


In [8]:
context.show(schema_gen.outputs['schema'])


,Type,Presence,Valency,Domain
Feature name,,,,
'age',INT,required,,-
'ca',INT,required,,-
'chol',INT,required,,-
'cp',INT,required,,-
'exang',INT,required,,-
'fbs',INT,required,,-
'oldpeak',FLOAT,required,,-
'restecg',INT,required,,-
'sex',INT,required,,-


## 6. Komponen 4: ExampleValidator (Data Validation & Anomaly Detection)
`ExampleValidator` memvalidasi data berdasarkan skema yang dihasilkan oleh `SchemaGen`. Komponen ini memeriksa apakah terdapat anomali, perbedaan format, nilai di luar rentang, atau kolom yang hilang.


In [9]:
from tfx.components import ExampleValidator

example_validator = ExampleValidator(
    statistics=statistics_gen.outputs['statistics'],
    schema=schema_gen.outputs['schema']
)
context.run(example_validator)


ExecutionResult(
    component_id: ExampleValidator
    execution_id: 14
    outputs:
        anomalies: OutputChannel(artifact_type=ExampleAnomalies, producer_component_id=ExampleValidator, output_key=anomalies, additional_properties={}, additional_custom_properties={}, _input_trigger=None)

Menampilkan hasil validasi anomali data (tabel kosong menunjukkan bahwa tidak ditemukan anomali pada dataset):


In [10]:
context.show(example_validator.outputs['anomalies'])


## 7. Komponen 5: Transform (Feature Engineering)
Komponen `Transform` melakukan preprocessing data menggunakan TensorFlow Transform (TFT). Keunggulan TFT adalah menghasilkan graf transformasi yang disertakan ke dalam model, sehingga menjamin konsistensi preprocessing data pada fase training dan inferensi (mencegah training-serving skew).
- Fitur numerik diskalakan menggunakan `tft.scale_to_z_score`.
- Fitur kategorikal dan label target dikonversi ke tipe `tf.int64`.


In [11]:
TRANSFORM_MODULE_FILE = os.path.join("modules", "heart_disease_transform.py")


In [12]:
%%writefile {TRANSFORM_MODULE_FILE}
import tensorflow as tf
import tensorflow_transform as tft

NUMERICAL_FEATURES = ["age", "trestbps", "chol", "thalach", "oldpeak"]
CATEGORICAL_FEATURES = ["sex", "cp", "fbs", "restecg", "exang", "slope", "ca", "thal"]
LABEL_KEY = "target"

def transformed_name(key: str) -> str:
    """Generate transformed feature name."""
    return f"{key}_xf"

def preprocessing_fn(inputs):
    """Preprocess input features into transformed features."""
    outputs = {}
    
    # Scale numerical features using z-score normalization
    for feature in NUMERICAL_FEATURES:
        outputs[transformed_name(feature)] = tft.scale_to_z_score(inputs[feature])
        
    # Cast categorical features to int64
    for feature in CATEGORICAL_FEATURES:
        outputs[transformed_name(feature)] = tf.cast(inputs[feature], tf.int64)
        
    # Target label
    outputs[transformed_name(LABEL_KEY)] = tf.cast(inputs[LABEL_KEY], tf.int64)
    
    return outputs


Overwriting modules\heart_disease_transform.py


Menjalankan komponen `Transform`:


In [13]:
from tfx.components import Transform

transform = Transform(
    examples=example_gen.outputs['examples'],
    schema=schema_gen.outputs['schema'],
    module_file=os.path.abspath(TRANSFORM_MODULE_FILE)
)
context.run(transform)


INFO:tensorflow:Assets written to: pipeline_root\Transform\transform_graph\15\.temp_path\tftransform_tmp\4f17599ca3a64920bd667b084c93bd5c\assets


INFO:tensorflow:Assets written to: pipeline_root\Transform\transform_graph\15\.temp_path\tftransform_tmp\4f17599ca3a64920bd667b084c93bd5c\assets


INFO:tensorflow:struct2tensor is not available.


INFO:tensorflow:struct2tensor is not available.


INFO:tensorflow:tensorflow_decision_forests is not available.


...

INFO:tensorflow:tensorflow_decision_forests is not available.


INFO:tensorflow:tensorflow_decision_forests is not available.


INFO:tensorflow:tensorflow_text is not available.


INFO:tensorflow:tensorflow_text is not available.


ExecutionResult(
    component_id: Transform
    execution_id: 15
    outputs:
        transform_graph: OutputChannel(artifact_type=TransformGraph, producer_component_id=Transform, output_key=transform_graph, additional_properties={}, additional_custom_properties={}, _input_trigger=None
        transformed_examples: OutputChannel(artifact_type=Examples, producer_component_id=Transform, output_key=transformed_examples, additional_properties={}, additional_custom_properties={}, _input_trigger=None
        updated_analyzer_cache: OutputChannel(artifact_type=TransformCache, producer_component_id=Transform, output_key=updated_analyzer_cache, additional_properties={}, additional_custom_properties={}, _input_trigger=None
        pre_transform_schema: OutputChannel(artifact_type=Schema, producer_component_id=Transform, output_key=pre_transform_schema, additional_properties={}, additional_custom_properties={}, _input_trigger=None
        pre_transform_stats: OutputChannel(artifact_type=ExampleStatistics, producer_component_id=Transform, output_key=pre_transform_stats, additional_properties={}, additional_custom_properties={}, _input_trigger=None
        post_transform_schema: OutputChannel(artifact_type=Schema, producer_component_id=Transform, output_key=post_transform_schema, additional_properties={}, additional_custom_properties={}, _input_trigger=None
        post_transform_stats: OutputChannel(artifact_type=ExampleStatistics, producer_component_id=Transform, output_key=post_transform_stats, additional_properties={}, additional_custom_properties={}, _input_trigger=None
        post_transform_anomalies: OutputChannel(artifact_type=ExampleAnomalies, producer_component_id=Transform, output_key=post_transform_anomalies, additional_properties={}, additional_custom_properties={}, _input_trigger=None)

## 8. Komponen Tambahan (Saran 1 - Bintang 5): Tuner (Hyperparameter Tuning)
Untuk meraih penilaian **Bintang 5**, kita menambahkan komponen `Tuner` yang memanfaatkan **KerasTuner** untuk mencari kombinasi hyperparameter terbaik secara otomatis (seperti jumlah unit neuron, dropout rate, dan learning rate optimizer).


In [14]:
TUNER_MODULE_FILE = os.path.join("modules", "heart_disease_tuner.py")


In [15]:
%%writefile {TUNER_MODULE_FILE}
from typing import NamedTuple, Dict, Text, Any
import keras_tuner as kt
import tensorflow as tf
import tensorflow_transform as tft
from tfx.components.trainer.fn_args_utils import FnArgs

NUMERICAL_FEATURES = ["age", "trestbps", "chol", "thalach", "oldpeak"]
CATEGORICAL_FEATURES = ["sex", "cp", "fbs", "restecg", "exang", "slope", "ca", "thal"]
LABEL_KEY = "target"

def transformed_name(key: str) -> str:
    """Generate transformed feature name."""
    return f"{key}_xf"

def gzip_reader_fn(filenames):
    """Small utility for reading gzip-compressed TFRecord files."""
    return tf.data.TFRecordDataset(filenames, compression_type="GZIP")

def input_fn(file_pattern, tf_transform_output, batch_size=32):
    """Generate batched TFRecord dataset for training and evaluation."""
    transformed_feature_spec = (
        tf_transform_output.transformed_feature_spec().copy()
    )
    dataset = tf.data.experimental.make_batched_features_dataset(
        file_pattern=file_pattern,
        batch_size=batch_size,
        features=transformed_feature_spec,
        reader=gzip_reader_fn,
        num_epochs=None,
        label_key=transformed_name(LABEL_KEY)
    )
    return dataset

def model_builder(hp):
    """Build Keras model with tunable hyperparameters."""
    input_features = []
    float_inputs = []
    
    for feature in NUMERICAL_FEATURES:
        inp = tf.keras.Input(shape=(1,), name=transformed_name(feature), dtype=tf.float32)
        input_features.append(inp)
        float_inputs.append(inp)
        
    for feature in CATEGORICAL_FEATURES:
        inp = tf.keras.Input(shape=(1,), name=transformed_name(feature), dtype=tf.int64)
        input_features.append(inp)
        float_inputs.append(tf.cast(inp, tf.float32))
        
    concat = tf.keras.layers.concatenate(float_inputs)
    
    hp_units_1 = hp.Int("units_1", min_value=32, max_value=128, step=32, default=64)
    hp_units_2 = hp.Int("units_2", min_value=16, max_value=64, step=16, default=32)
    hp_dropout = hp.Float("dropout", min_value=0.1, max_value=0.3, step=0.1, default=0.2)
    hp_learning_rate = hp.Choice("learning_rate", values=[1e-2, 1e-3, 1e-4], default=1e-3)
    
    x = tf.keras.layers.Dense(hp_units_1, activation="relu")(concat)
    x = tf.keras.layers.Dropout(hp_dropout)(x)
    x = tf.keras.layers.Dense(hp_units_2, activation="relu")(x)
    x = tf.keras.layers.Dropout(hp_dropout)(x)
    outputs = tf.keras.layers.Dense(1, activation="sigmoid")(x)
    
    model = tf.keras.Model(inputs=input_features, outputs=outputs)
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=hp_learning_rate),
        loss="binary_crossentropy",
        metrics=["accuracy", tf.keras.metrics.AUC(name="auc")]
    )
    return model

TunerFnResult = NamedTuple("TunerFnResult", [
    ("tuner", kt.engine.base_tuner.BaseTuner),
    ("fit_kwargs", Dict[Text, Any])
])

def tuner_fn(fn_args: FnArgs) -> TunerFnResult:
    """Build the tuner using KerasTuner RandomSearch."""
    transform_graph_path = getattr(fn_args, "transform_graph_path", None) or getattr(fn_args, "transform_output", None)
    tf_transform_output = tft.TFTransformOutput(transform_graph_path)
    
    train_dataset = input_fn(fn_args.train_files, tf_transform_output, batch_size=32)
    eval_dataset = input_fn(fn_args.eval_files, tf_transform_output, batch_size=32)
    
    tuner = kt.RandomSearch(
        hypermodel=model_builder,
        objective=kt.Objective("val_auc", direction="max"),
        max_trials=3,
        directory=fn_args.working_dir,
        project_name="heart_disease_tuning"
    )
    
    train_steps = fn_args.train_steps or 20
    eval_steps = fn_args.eval_steps or 5
    
    return TunerFnResult(
        tuner=tuner,
        fit_kwargs={
            "callbacks": [tf.keras.callbacks.EarlyStopping(monitor="val_auc", mode="max", patience=3)],
            "x": train_dataset,
            "validation_data": eval_dataset,
            "steps_per_epoch": train_steps,
            "validation_steps": eval_steps,
            "epochs": 5
        }
    )


Overwriting modules\heart_disease_tuner.py


Menjalankan komponen `Tuner` dengan split train dan eval:


In [16]:
from tfx.components import Tuner
from tfx.proto import trainer_pb2

tuner = Tuner(
    module_file=os.path.abspath(TUNER_MODULE_FILE),
    examples=transform.outputs['transformed_examples'],
    transform_graph=transform.outputs['transform_graph'],
    schema=schema_gen.outputs['schema'],
    train_args=trainer_pb2.TrainArgs(splits=['train'], num_steps=20),
    eval_args=trainer_pb2.EvalArgs(splits=['eval'], num_steps=5)
)
context.run(tuner)


Trial 3 Complete [00h 00m 01s]
val_auc: 0.8662762641906738

Best val_auc So Far: 0.8803608417510986
Total elapsed time: 00h 00m 04s
Results summary
Results in pipeline_root\.temp\16\heart_disease_tuning
Showing 10 best trials
Objective(name="val_auc", direction="max")

Trial 0 summary
Hyperparameters:
units_1: 96
units_2: 16
dropout: 0.1
learning_rate: 0.01
Score: 0.8803608417510986

Trial 1 summary
Hyperparameters:
units_1: 96
units_2: 32
dropout: 0.1
learning_rate: 0.001
Score: 0.8667139410972595

Trial 2 summary
Hyperparameters:
units_1: 32
units_2: 64
dropout: 0.2
learning_rate: 0.01
Score: 0.8662762641906738


ExecutionResult(
    component_id: Tuner
    execution_id: 16
    outputs:
        best_hyperparameters: OutputChannel(artifact_type=HyperParameters, producer_component_id=Tuner, output_key=best_hyperparameters, additional_properties={}, additional_custom_properties={}, _input_trigger=None
        tuner_results: OutputChannel(artifact_type=TunerResults, producer_component_id=Tuner, output_key=tuner_results, additional_properties={}, additional_custom_properties={}, _input_trigger=None)

## 9. Komponen 6: Trainer (Model Training dengan Best Hyperparameters)
Komponen `Trainer` menggunakan `best_hyperparameters` yang dihasilkan oleh komponen `Tuner` untuk melatih model akhir Deep Neural Network secara optimal.
Model diekspor lengkap dengan serving signature (`serving_default`) yang menerima serialized `tf.train.Example` dan layer preprocessing TFT.


In [17]:
TRAINER_MODULE_FILE = os.path.join("modules", "heart_disease_trainer.py")


In [18]:
%%writefile {TRAINER_MODULE_FILE}
import os
import tensorflow as tf
import tensorflow_transform as tft
import keras_tuner as kt
from tfx.components.trainer.fn_args_utils import FnArgs

NUMERICAL_FEATURES = ["age", "trestbps", "chol", "thalach", "oldpeak"]
CATEGORICAL_FEATURES = ["sex", "cp", "fbs", "restecg", "exang", "slope", "ca", "thal"]
LABEL_KEY = "target"

def transformed_name(key: str) -> str:
    """Generate transformed feature name."""
    return f"{key}_xf"

def gzip_reader_fn(filenames):
    """Small utility for reading gzip-compressed TFRecord files."""
    return tf.data.TFRecordDataset(filenames, compression_type="GZIP")

def input_fn(file_pattern, tf_transform_output, batch_size=32):
    """Generate batched TFRecord dataset for training and evaluation."""
    transformed_feature_spec = (
        tf_transform_output.transformed_feature_spec().copy()
    )
    dataset = tf.data.experimental.make_batched_features_dataset(
        file_pattern=file_pattern,
        batch_size=batch_size,
        features=transformed_feature_spec,
        reader=gzip_reader_fn,
        num_epochs=None,
        label_key=transformed_name(LABEL_KEY)
    )
    return dataset

def get_model(hp=None):
    """Build Keras Deep Neural Network model, optionally with tuned hyperparameters."""
    input_features = []
    float_inputs = []
    
    for feature in NUMERICAL_FEATURES:
        inp = tf.keras.Input(shape=(1,), name=transformed_name(feature), dtype=tf.float32)
        input_features.append(inp)
        float_inputs.append(inp)
        
    for feature in CATEGORICAL_FEATURES:
        inp = tf.keras.Input(shape=(1,), name=transformed_name(feature), dtype=tf.int64)
        input_features.append(inp)
        float_inputs.append(tf.cast(inp, tf.float32))
        
    concat = tf.keras.layers.concatenate(float_inputs)
    
    if hp is not None and hasattr(hp, "get"):
        units_1 = hp.get("units_1") if "units_1" in hp.values else 64
        units_2 = hp.get("units_2") if "units_2" in hp.values else 32
        dropout = hp.get("dropout") if "dropout" in hp.values else 0.2
        lr = hp.get("learning_rate") if "learning_rate" in hp.values else 0.001
    else:
        units_1 = 64
        units_2 = 32
        dropout = 0.2
        lr = 0.001
        
    x = tf.keras.layers.Dense(units_1, activation="relu")(concat)
    x = tf.keras.layers.Dropout(dropout)(x)
    x = tf.keras.layers.Dense(units_2, activation="relu")(x)
    x = tf.keras.layers.Dropout(dropout)(x)
    outputs = tf.keras.layers.Dense(1, activation="sigmoid")(x)
    
    model = tf.keras.Model(inputs=input_features, outputs=outputs)
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=lr),
        loss="binary_crossentropy",
        metrics=["accuracy", tf.keras.metrics.AUC(name="auc")]
    )
    return model

def _get_serve_tf_examples_fn(model, tf_transform_output):
    """Returns a function that parses raw serialized tf.Example for serving."""
    model.tft_layer = tf_transform_output.transform_features_layer()

    @tf.function
    def serve_tf_examples_fn(serialized_tf_examples):
        feature_spec = tf_transform_output.raw_feature_spec()
        feature_spec.pop(LABEL_KEY, None)
        parsed_features = tf.io.parse_example(serialized_tf_examples, feature_spec)
        transformed_features = model.tft_layer(parsed_features)
        return model(transformed_features)

    return serve_tf_examples_fn

def run_fn(fn_args: FnArgs):
    """Train the model based on given args."""
    tf_transform_output = tft.TFTransformOutput(fn_args.transform_output)
    
    train_dataset = input_fn(fn_args.train_files, tf_transform_output, batch_size=32)
    eval_dataset = input_fn(fn_args.eval_files, tf_transform_output, batch_size=32)
    
    hp = None
    if fn_args.hyperparameters:
        hp = kt.HyperParameters.from_config(fn_args.hyperparameters)
        
    model = get_model(hp)
    
    tensorboard_callback = tf.keras.callbacks.TensorBoard(
        log_dir=fn_args.model_run_dir, update_freq="batch"
    )
    
    train_steps = fn_args.train_steps or 20
    eval_steps = fn_args.eval_steps or 5
    
    model.fit(
        train_dataset,
        steps_per_epoch=train_steps,
        validation_data=eval_dataset,
        validation_steps=eval_steps,
        callbacks=[tensorboard_callback],
        epochs=10
    )
    
    signatures = {
        "serving_default": _get_serve_tf_examples_fn(
            model, tf_transform_output
        ).get_concrete_function(
            tf.TensorSpec(shape=[None], dtype=tf.string, name="examples")
        )
    }
    
    model.save(fn_args.serving_model_dir, save_format="tf", signatures=signatures)


Overwriting modules\heart_disease_trainer.py


Menjalankan komponen `Trainer` dengan memasukkan `hyperparameters=tuner.outputs['best_hyperparameters']`:


In [19]:
from tfx.components import Trainer
from tfx.proto import trainer_pb2

trainer = Trainer(
    module_file=os.path.abspath(TRAINER_MODULE_FILE),
    examples=transform.outputs['transformed_examples'],
    transform_graph=transform.outputs['transform_graph'],
    schema=schema_gen.outputs['schema'],
    hyperparameters=tuner.outputs['best_hyperparameters'],
    train_args=trainer_pb2.TrainArgs(splits=['train'], num_steps=20),
    eval_args=trainer_pb2.EvalArgs(splits=['eval'], num_steps=5)
)
context.run(trainer)


Epoch 1/10


 1/20 [>.............................] - ETA: 8s - loss: 0.7198 - accuracy: 0.4688 - auc: 0.4588

18/20 [==========================>...] - ETA: 0s - loss: 0.4333 - accuracy: 0.7847 - auc: 0.8771

20/20 [==============================] - 1s 12ms/step - loss: 0.4130 - accuracy: 0.7984 - auc: 0.8879 - val_loss: 0.5881 - val_accuracy: 0.7688 - val_auc: 0.8502


Epoch 2/10


...

INFO:tensorflow:tensorflow_text is not available.


INFO:tensorflow:tensorflow_text is not available.


INFO:tensorflow:Assets written to: pipeline_root\Trainer\model\17\Format-Serving\assets


INFO:tensorflow:Assets written to: pipeline_root\Trainer\model\17\Format-Serving\assets


ExecutionResult(
    component_id: Trainer
    execution_id: 17
    outputs:
        model: OutputChannel(artifact_type=Model, producer_component_id=Trainer, output_key=model, additional_properties={}, additional_custom_properties={}, _input_trigger=None
        model_run: OutputChannel(artifact_type=ModelRun, producer_component_id=Trainer, output_key=model_run, additional_properties={}, additional_custom_properties={}, _input_trigger=None)

## 10. Komponen 7: Resolver (Model Baseline Resolver)
Komponen `Resolver` (menggunakan `LatestBlessedModelResolver`) bertugas menentukan model terbaik sebelumnya yang telah disetujui (blessed) untuk dijadikan acuan komparasi baseline oleh komponen `Evaluator`.


In [20]:
from tfx.dsl.components.common.resolver import Resolver
from tfx.dsl.experimental.latest_blessed_model_resolver import LatestBlessedModelResolver
from tfx.types import Channel
from tfx.types.standard_artifacts import Model, ModelBlessing

model_resolver = Resolver(
    strategy_class=LatestBlessedModelResolver,
    model=Channel(type=Model),
    model_blessing=Channel(type=ModelBlessing)
).with_id('latest_blessed_model_resolver')

context.run(model_resolver)


ExecutionResult(
    component_id: latest_blessed_model_resolver
    execution_id: 18
    outputs:
        model: OutputChannel(artifact_type=Model, producer_component_id=latest_blessed_model_resolver, output_key=model, additional_properties={}, additional_custom_properties={}, _input_trigger=None
        model_blessing: OutputChannel(artifact_type=ModelBlessing, producer_component_id=latest_blessed_model_resolver, output_key=model_blessing, additional_properties={}, additional_custom_properties={}, _input_trigger=None)

## 11. Komponen 8: Evaluator (Model Evaluation & Analysis)
Komponen `Evaluator` menggunakan TensorFlow Model Analysis (TFMA) untuk mengevaluasi kualitas model secara mendalam.
- Menghitung metrik AUC, BinaryAccuracy, True Positives, False Positives, dsb.
- Menerapkan slicing pada keseluruhan dataset (overall) dan slice fitur spesifik (misal fitur jenis kelamin `sex`).
- Memvalidasi threshold kelayakan model (misal BinaryAccuracy minimal 0.5) agar model layak mendapatkan blessing untuk dideploy.


In [21]:
import tensorflow_model_analysis as tfma
from tfx.components import Evaluator

eval_config = tfma.EvalConfig(
    model_specs=[tfma.ModelSpec(label_key="target")],
    slicing_specs=[
        tfma.SlicingSpec(),
        tfma.SlicingSpec(feature_keys=["sex"])
    ],
    metrics_specs=[
        tfma.MetricsSpec(metrics=[
            tfma.MetricConfig(class_name="AUC"),
            tfma.MetricConfig(class_name="FalsePositives"),
            tfma.MetricConfig(class_name="TruePositives"),
            tfma.MetricConfig(class_name="FalseNegatives"),
            tfma.MetricConfig(class_name="TrueNegatives"),
            tfma.MetricConfig(
                class_name="BinaryAccuracy",
                threshold=tfma.MetricThreshold(
                    value_threshold=tfma.GenericValueThreshold(
                        lower_bound={"value": 0.5}
                    ),
                    change_threshold=tfma.GenericChangeThreshold(
                        direction=tfma.MetricDirection.HIGHER_IS_BETTER,
                        absolute={"value": -1e-10}
                    )
                )
            )
        ])
    ]
)

evaluator = Evaluator(
    examples=example_gen.outputs['examples'],
    model=trainer.outputs['model'],
    baseline_model=model_resolver.outputs['model'],
    eval_config=eval_config
)
context.run(evaluator)


Instructions for updating:
Use eager execution and: 
`tf.data.TFRecordDataset(path)`


Instructions for updating:
Use eager execution and: 
`tf.data.TFRecordDataset(path)`


ExecutionResult(
    component_id: Evaluator
    execution_id: 19
    outputs:
        evaluation: OutputChannel(artifact_type=ModelEvaluation, producer_component_id=Evaluator, output_key=evaluation, additional_properties={}, additional_custom_properties={}, _input_trigger=None
        blessing: OutputChannel(artifact_type=ModelBlessing, producer_component_id=Evaluator, output_key=blessing, additional_properties={}, additional_custom_properties={}, _input_trigger=None)

Memeriksa status artefak blessing model (BLESSED menandakan model lolos threshold):


In [22]:
blessing_artifact = evaluator.outputs['blessing'].get()[0]
print("Blessing Artifact URI:", blessing_artifact.uri)
is_blessed = os.path.exists(os.path.join(blessing_artifact.uri, 'BLESSED'))
print("Status Blessing Model:", "BLESSED (Lolos)" if is_blessed else "NOT BLESSED (Gagal)")


Blessing Artifact URI: pipeline_root\Evaluator\blessing\19
Status Blessing Model: BLESSED (Lolos)


Visualisasi evaluasi TFMA:


In [23]:
context.show(evaluator.outputs['evaluation'])


SlicingMetricsViewer(config={'weightedExamplesColumn': 'example_count'}, data=[{'slice': 'Overall', 'metrics':…

## 12. Komponen 9: Pusher (Model Deployment)
Komponen `Pusher` memverifikasi hasil blessing dari `Evaluator`. Jika model berhasil di-bless, `Pusher` secara otomatis menyalin artefak model SavedModel ke direktori serving (`serving_model_dir`) yang siap dikonsumsi oleh TensorFlow Serving atau aplikasi produksi.


In [24]:
from tfx.components import Pusher
from tfx.proto import pusher_pb2

pusher = Pusher(
    model=trainer.outputs['model'],
    model_blessing=evaluator.outputs['blessing'],
    push_destination=pusher_pb2.PushDestination(
        filesystem=pusher_pb2.PushDestination.Filesystem(
            base_directory=os.path.abspath(SERVING_MODEL_DIR)
        )
    )
)
context.run(pusher)


ExecutionResult(
    component_id: Pusher
    execution_id: 20
    outputs:
        pushed_model: OutputChannel(artifact_type=PushedModel, producer_component_id=Pusher, output_key=pushed_model, additional_properties={}, additional_custom_properties={}, _input_trigger=None)

Memeriksa isi direktori serving model yang telah di-deploy:


In [25]:
pushed_versions = os.listdir(SERVING_MODEL_DIR)
print(f"Model berhasil dideploy ke {SERVING_MODEL_DIR}!")
print("Versi model yang tersedia:", pushed_versions)


Model berhasil dideploy ke serving_model_dir!
Versi model yang tersedia: ['1789440851', '1789462532']


---
## Kesimpulan
Pipeline Machine Learning menggunakan TensorFlow Extended (TFX) dengan seluruh komponen wajib dan saran tambahan (Bintang 5) telah berhasil dieksekusi secara lengkap:
1. **ExampleGen**: Berhasil mengonversi dan membagi dataset ke dalam format TFRecord.
2. **StatisticsGen**: Berhasil menghitung statistik data latih dan evaluasi secara terperinci.
3. **SchemaGen**: Berhasil membuat skema fitur yang merepresentasikan tipe dan domain data.
4. **ExampleValidator**: Memvalidasi data dan memastikan tidak ada anomali atau data drift.
5. **Transform**: Melakukan scaling numerik z-score dan transformasi fitur secara konsisten.
6. **Tuner (Saran 1)**: Menemukan konfigurasi hyperparameter optimal menggunakan KerasTuner RandomSearch.
7. **Trainer**: Melatih model DNN dengan hyperparameter terbaik dan mengekspor SavedModel bertanda tangan serving.
8. **Resolver**: Berhasil mendeteksi baseline model terdahulu.
9. **Evaluator**: Mengevaluasi performa model dengan TFMA (akurasi, AUC, fairness slicing fitur jenis kelamin) dan meloloskan model (*BLESSED*).
10. **Pusher**: Mendistribusikan model ke direktori `serving_model_dir` siap pakai untuk produksi.
